# Edit bgsub masks

In [ ]:

import bioio_ome_tiff
import bioio_tifffile


%matplotlib notebook
%matplotlib inline





In [ ]:
input_dirpath = Path(input())

In [ ]:
output_dirpath = input_dirpath.parent / (input_dirpath.name + '_edited')
figs_dirpath = output_dirpath / dn.figs_dirname
figs_dirpath.mkdir(exist_ok=True, parents=True)

In [ ]:
seg_cmpch = 2
seg_caaxch = 3

imgpaths = [path for path in input_dirpath.glob('*.ome.tif')]
imgpaths.sort()

for imgpath in tqdm(imgpaths):
    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data

    seg_cmp = (img[:, seg_cmpch, np.newaxis, :, :, :] > 0).astype('bool')
    seg_caax = (img[:, seg_caaxch, np.newaxis, :, :, :] > 0).astype('bool')

    seg_cell = (seg_cmp | seg_caax)
    seg_cell = am.remove_unconnected_areas(seg_cell)
    seg_cell = am.remove_small_holes(seg_cell, area_threshold=5)
    
    # mask caax channel with cell channel
    seg_caax = am.mask_img(seg_caax, seg_cell)
    seg_caax = am.remove_small_holes(seg_caax, area_threshold=5)
    seg_caax = am.remove_small_objects(seg_caax, min_size=5)
    

    # get cmp areas from newly calculated cellmask and caax
    seg_cmp = (seg_cell & ~seg_caax)
    seg_cmp_prev = seg_cmp
    seg_cmp = am.remove_cmp_near_cellborders(seg_cell, seg_cmp, iter_dil=5)

    seg_cell = (seg_cmp | seg_caax)
    seg_cell = am.remove_unconnected_areas(seg_cell)    
    seg_caax = (seg_caax & seg_cell)
    seg_cmp = (seg_cmp & seg_cell)
    
    # create and save compaction figure
    compacted_val = 255
    caax_val = 100
    figname =  imgpath.name.replace('.ome.tif', '.tif')
    comp_fig = seg_cmp * compacted_val + seg_caax * caax_val
    comp_fig = comp_fig.astype('uint8')
    comp_fig = comp_fig.squeeze()
    OmeTiffWriter.save(comp_fig, figs_dirpath / figname)

    # create img stack with binary compacted area, caax, and cell
    max_pix_val = np.iinfo(img_file.dtype).max
    comp_stack = np.concatenate([img[:, :seg_cmpch, :, :, :], seg_cmp, seg_caax], axis=1)
    comp_stack[:, [seg_cmpch, seg_caaxch], :, :, :] = (comp_stack[:, [seg_cmpch, seg_caaxch], :, :, :] * max_pix_val)
    comp_stack = comp_stack.astype(np.uint16)

    # # save img with binary caax and cellch
    output_imgpath = output_dirpath / imgpath.name
    ome_metadata = utils.construct_ome_metadata(comp_stack, img_file)
    OmeTiffWriter.save(comp_stack, output_imgpath, ome_xml=ome_metadata)

    
